# Alpha-Beta Analysis Notebook (Rewritten)

Notebook này đã được viết lại để tránh các lỗi trước đó:

- giữ **dữ liệu gốc** (`*_df_raw`) để đếm `reject_reason`
- **không xóa dòng chỉ vì `raw_m` là NaN**
- có đầy đủ các hàm:
  - `load_log_csv`
  - `run_alpha_beta`
  - `auto_detect_switch_windows`
  - `analyze_switch_windows`
  - `score_static`
  - `score_switch`
- chọn **top-K** từ bài tripod rồi mới sweep trên bài switching
- export đầy đủ các file CSV kết quả

In [ ]:
# ====== Upload file (có thể bỏ qua nếu file đã có sẵn trên Colab) ======
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
except Exception:
    print("Không chạy trên Colab hoặc bạn có thể bỏ qua cell này nếu file đã có sẵn.")

In [ ]:
# ====== Imports ======
import os
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
# ====== Cấu hình người dùng ======
TRIPOD_CSV = "Tripod_531m_baseline.csv"
SWITCH_CSV = "Switching_7m_525m_baseline.csv"

TRIPOD_TRUTH_M = 531.0
LOW_TARGET_M = 7.0
HIGH_TARGET_M = 525.0

# Nếu muốn tự chỉ định từng lần đổi mục tiêu, điền vào đây.
SWITCH_WINDOWS = []

# grid search trên tripod
ALPHA_LIST = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
BETA_LIST  = [0.00, 0.01, 0.02, 0.03, 0.05]
STATIC_GATE_LIST = [6.0, 8.0, 10.0]

# từ tripod lấy top-K cấu hình tốt nhất
TOP_K_STATIC = 5

# sweep thêm ở bài switching
SWITCH_GATE_LIST = [8.0, 10.0, 12.0, 15.0]
SWITCH_MAX_REJECT_LIST = [3, 5, 7]
SWITCH_SETTLING_TOL_M = 5.0

print("Files in cwd:", sorted(os.listdir("."))[:100])

In [ ]:
# ====== Helper ======
def safe_mean(x):
    if x is None:
        return np.nan
    try:
        s = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
        return float(s.mean()) if len(s) else np.nan
    except Exception:
        try:
            return float(x)
        except Exception:
            return np.nan

def ensure_columns(df, cols_with_default):
    out = df.copy()
    for col, default in cols_with_default.items():
        if col not in out.columns:
            out[col] = default
    return out

In [ ]:
# ====== Load và chuẩn hóa CSV ======
def load_log_csv(path):
    df = pd.read_csv(path)

    # Chuẩn hóa tên cột phổ biến
    rename_map = {}
    for src, dst in [
        ("dev_ts_ms", "timestamp_ms"),
        ("t_ms", "timestamp_ms"),
        ("rawDistanceM", "raw_m"),
        ("filteredDistanceM", "est_m"),
        ("predictedDistanceM", "predicted_m"),
        ("rangeRateMps", "rate_mps"),
        ("residualM", "residual_m"),
        ("rejectedByGate", "rejected_by_gate"),
        ("measStatus", "meas_status"),
        ("trackState", "track_state"),
    ]:
        if src in df.columns and dst not in df.columns:
            rename_map[src] = dst
    if rename_map:
        df = df.rename(columns=rename_map)

    df = ensure_columns(df, {
        "timestamp_ms": np.nan,
        "raw_m": np.nan,
        "est_m": np.nan,
        "predicted_m": np.nan,
        "rate_mps": np.nan,
        "residual_m": np.nan,
        "rejected_by_gate": 0,
        "reject_reason": 0,
        "meas_status": np.nan,
        "track_state": np.nan,
        "test_id": np.nan,
        "mode": np.nan,
    })

    # Ép kiểu
    numeric_cols = [
        "timestamp_ms", "raw_m", "est_m", "predicted_m", "rate_mps",
        "residual_m", "rejected_by_gate", "reject_reason",
        "meas_status", "track_state", "test_id", "mode"
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Chỉ bỏ dòng không có timestamp
    df = df.dropna(subset=["timestamp_ms"]).copy()
    df["timestamp_ms"] = df["timestamp_ms"].astype(int)
    df = df.sort_values("timestamp_ms").reset_index(drop=True)

    # Trục thời gian
    df["t_s"] = (df["timestamp_ms"] - df["timestamp_ms"].iloc[0]) / 1000.0

    # raw hữu hạn hay không
    df["raw_finite"] = np.isfinite(df["raw_m"])

    # sensor valid: chỉ valid nếu không bị reject ở sensor/parser/gate và raw hữu hạn
    df["reject_reason"] = df["reject_reason"].fillna(0).astype(int)
    df["rejected_by_gate"] = df["rejected_by_gate"].fillna(0).astype(int)
    df["sensor_valid"] = (df["reject_reason"] == 0) & df["raw_finite"]

    # baseline_m: ưu tiên est_m nếu có, nếu không thì fallback raw_m
    df["baseline_m"] = df["est_m"]
    df.loc[~np.isfinite(df["baseline_m"]), "baseline_m"] = df.loc[~np.isfinite(df["baseline_m"]), "raw_m"]

    return df

tripod_df_raw = load_log_csv(TRIPOD_CSV)
switch_df_raw = load_log_csv(SWITCH_CSV)

tripod_df = tripod_df_raw.copy()
switch_df = switch_df_raw.copy()

print(f"TRIPOD_CSV = {TRIPOD_CSV}")
print(f"SWITCH_CSV = {SWITCH_CSV}")
print(f"Tripod rows (raw): {len(tripod_df_raw)}")
print(f"Switch rows (raw): {len(switch_df_raw)}")

print("\nTripod reject_reason counts:")
print(tripod_df_raw["reject_reason"].value_counts(dropna=False).sort_index())

print("\nSwitch reject_reason counts:")
print(switch_df_raw["reject_reason"].value_counts(dropna=False).sort_index())

display(tripod_df_raw.head())
display(switch_df_raw.head())

In [ ]:
# ====== Plot reject reason theo thời gian ======
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=False)

axes[0].step(tripod_df_raw["t_s"], tripod_df_raw["reject_reason"], where="post")
axes[0].set_title("Tripod reject_reason over time")
axes[0].set_ylabel("reject_reason")

axes[1].step(switch_df_raw["t_s"], switch_df_raw["reject_reason"], where="post")
axes[1].set_title("Switching reject_reason over time")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("reject_reason")

plt.tight_layout()
plt.show()

In [ ]:
# ====== Alpha-Beta core ======
@dataclass
class ABConfig:
    alpha: float = 0.10
    beta: float = 0.00
    gate_threshold_m: float = 8.0
    max_reject: int = 5
    default_dt_s: float = 0.15
    min_dt_s: float = 1e-3
    max_dt_s: float = 2.0
    reinit_on_switch: bool = True

def run_alpha_beta(df, cfg: ABConfig):
    out = df.copy()

    # cột output
    out["ab_est_m"] = np.nan
    out["ab_predicted_m"] = np.nan
    out["ab_rate_mps"] = np.nan
    out["ab_residual_m"] = np.nan
    out["ab_rejected_by_gate"] = 0

    x = np.nan
    v = 0.0
    last_ts = None
    reject_count = 0
    initialized = False

    for i, row in out.iterrows():
        ts_ms = row["timestamp_ms"]
        z = row["raw_m"]
        sensor_valid = bool(row["sensor_valid"]) if "sensor_valid" in out.columns else bool(np.isfinite(z))

        if last_ts is None:
            dt = cfg.default_dt_s
        else:
            dt = (ts_ms - last_ts) / 1000.0
            if not np.isfinite(dt):
                dt = cfg.default_dt_s
            dt = min(max(dt, cfg.min_dt_s), cfg.max_dt_s)

        if not initialized:
            if sensor_valid and np.isfinite(z):
                x = float(z)
                v = 0.0
                initialized = True
                out.at[i, "ab_est_m"] = x
                out.at[i, "ab_predicted_m"] = x
                out.at[i, "ab_rate_mps"] = v
                out.at[i, "ab_residual_m"] = 0.0
            last_ts = ts_ms
            continue

        # predict
        x_pred = x + v * dt
        v_pred = v

        out.at[i, "ab_predicted_m"] = x_pred

        if sensor_valid and np.isfinite(z):
            residual = float(z) - x_pred
            out.at[i, "ab_residual_m"] = residual

            # gate ở lớp tracker
            if abs(residual) > cfg.gate_threshold_m:
                out.at[i, "ab_rejected_by_gate"] = 1
                reject_count += 1

                if cfg.reinit_on_switch and reject_count >= cfg.max_reject:
                    x = float(z)
                    v = 0.0
                    reject_count = 0
                else:
                    x = x_pred
                    v = v_pred
            else:
                x = x_pred + cfg.alpha * residual
                if cfg.beta > 0 and dt > cfg.min_dt_s:
                    v = v_pred + (cfg.beta / dt) * residual
                else:
                    v = v_pred
                reject_count = 0
        else:
            # sensor invalid -> predict only
            x = x_pred
            v = v_pred

        out.at[i, "ab_est_m"] = x
        out.at[i, "ab_rate_mps"] = v

        last_ts = ts_ms

    return out

In [ ]:
# ====== Metric tripod ======
def summarize_tripod_baseline(df, truth_m):
    raw = pd.to_numeric(df["raw_m"], errors="coerce")
    baseline = pd.to_numeric(df["baseline_m"], errors="coerce")

    return pd.DataFrame([{
        "n_samples": len(df),
        "mean_fps": safe_mean(df["fps"]) if "fps" in df.columns else np.nan,
        "raw_mean": safe_mean(raw),
        "raw_sigma": float(raw.std(ddof=1)),
        "raw_bias": safe_mean(raw) - truth_m,
        "raw_rmse": float(np.sqrt(np.nanmean((raw - truth_m) ** 2))),
        "baseline_mean": safe_mean(baseline),
        "baseline_sigma": float(baseline.std(ddof=1)),
        "baseline_bias": safe_mean(baseline) - truth_m,
        "baseline_rmse": float(np.sqrt(np.nanmean((baseline - truth_m) ** 2))),
        "sensor_reject_ratio": float((df["reject_reason"] != 0).mean()),
        "tracker_gate_reject_ratio": float((df["rejected_by_gate"] != 0).mean()) if "rejected_by_gate" in df.columns else 0.0,
    }])

def score_static(ab_sigma, ab_bias, ab_rate_std, tracker_gate_reject_ratio):
    # score nhỏ hơn là tốt hơn
    ab_sigma = 1e6 if pd.isna(ab_sigma) else ab_sigma
    ab_bias = 1e6 if pd.isna(ab_bias) else abs(ab_bias)
    ab_rate_std = 1e6 if pd.isna(ab_rate_std) else ab_rate_std
    tracker_gate_reject_ratio = 1.0 if pd.isna(tracker_gate_reject_ratio) else tracker_gate_reject_ratio
    return float(
        1.0 * ab_sigma +
        0.5 * ab_bias +
        0.3 * ab_rate_std +
        0.2 * tracker_gate_reject_ratio
    )

baseline_summary_tripod = summarize_tripod_baseline(tripod_df, TRIPOD_TRUTH_M)
display(baseline_summary_tripod)

In [ ]:
# ====== Grid search trên tripod ======
rows = []

for a in ALPHA_LIST:
    for b in BETA_LIST:
        for g in STATIC_GATE_LIST:
            cfg = ABConfig(alpha=float(a), beta=float(b), gate_threshold_m=float(g), max_reject=5)
            ab_df = run_alpha_beta(tripod_df, cfg)

            baseline = pd.to_numeric(ab_df["baseline_m"], errors="coerce")
            ab_est = pd.to_numeric(ab_df["ab_est_m"], errors="coerce")
            ab_rate = pd.to_numeric(ab_df["ab_rate_mps"], errors="coerce")

            ab_sigma = float(ab_est.std(ddof=1))
            ab_bias = safe_mean(ab_est) - TRIPOD_TRUTH_M
            ab_rmse = float(np.sqrt(np.nanmean((ab_est - TRIPOD_TRUTH_M) ** 2)))
            ab_rate_std = float(ab_rate.std(ddof=1))
            ab_tracker_gate_reject_ratio = float((ab_df["ab_rejected_by_gate"] != 0).mean())
            sensor_reject_ratio = float((ab_df["reject_reason"] != 0).mean())

            rows.append({
                "alpha": float(a),
                "beta": float(b),
                "gate_m": float(g),
                "ab_sigma": ab_sigma,
                "ab_bias": ab_bias,
                "ab_rmse": ab_rmse,
                "ab_rate_std": ab_rate_std,
                "sensor_reject_ratio": sensor_reject_ratio,
                "ab_tracker_gate_reject_ratio": ab_tracker_gate_reject_ratio,
                "score_static": score_static(ab_sigma, ab_bias, ab_rate_std, ab_tracker_gate_reject_ratio),
            })

res_df = pd.DataFrame(rows).sort_values(
    ["score_static", "ab_sigma", "ab_rmse", "ab_rate_std", "ab_tracker_gate_reject_ratio"]
).reset_index(drop=True)

display(res_df.head(20))

In [ ]:
# ====== So sánh baseline vs alpha-beta trên tripod với cấu hình tốt nhất theo static ======
best_static = res_df.iloc[0]
best_static_cfg = ABConfig(
    alpha=float(best_static["alpha"]),
    beta=float(best_static["beta"]),
    gate_threshold_m=float(best_static["gate_m"]),
    max_reject=5,
)

tripod_ab = run_alpha_beta(tripod_df, best_static_cfg)

baseline = pd.to_numeric(tripod_ab["baseline_m"], errors="coerce")
ab_est = pd.to_numeric(tripod_ab["ab_est_m"], errors="coerce")
ab_rate = pd.to_numeric(tripod_ab["ab_rate_mps"], errors="coerce")

tripod_ab_compare = pd.DataFrame([{
    "baseline_sigma": float(baseline.std(ddof=1)),
    "ab_sigma": float(ab_est.std(ddof=1)),
    "baseline_rmse": float(np.sqrt(np.nanmean((baseline - TRIPOD_TRUTH_M) ** 2))),
    "ab_rmse": float(np.sqrt(np.nanmean((ab_est - TRIPOD_TRUTH_M) ** 2))),
    "ab_rate_std": float(ab_rate.std(ddof=1)),
    "sensor_reject_ratio": float((tripod_ab["reject_reason"] != 0).mean()),
    "ab_tracker_gate_reject_ratio": float((tripod_ab["ab_rejected_by_gate"] != 0).mean()),
}])

display(tripod_ab_compare)

plt.figure(figsize=(12, 4))
plt.plot(tripod_ab["t_s"], tripod_ab["raw_m"], label="raw")
plt.plot(tripod_ab["t_s"], tripod_ab["baseline_m"], label="baseline")
plt.plot(tripod_ab["t_s"], tripod_ab["ab_est_m"], label="alpha-beta")
plt.axhline(TRIPOD_TRUTH_M, linestyle="--", label="truth")
plt.title("Tripod static: raw vs baseline vs alpha-beta")
plt.xlabel("Time (s)")
plt.ylabel("Distance (m)")
plt.legend()
plt.show()

In [ ]:
# ====== Detect windows cho switching ======
def auto_detect_switch_windows(
    df,
    low_target_m,
    high_target_m,
    signal_col="raw_m",
    min_gap_s=2.0,
    near_tol_m=40.0,
    min_stable_samples=2,
):
    work = df.copy()
    work[signal_col] = pd.to_numeric(work[signal_col], errors="coerce")
    work = work.dropna(subset=["t_s"]).sort_values("t_s").reset_index(drop=True)

    def classify(x):
        if pd.isna(x):
            return "mid"
        if abs(x - low_target_m) <= near_tol_m:
            return "low"
        if abs(x - high_target_m) <= near_tol_m:
            return "high"
        return "mid"

    work["state_raw"] = work[signal_col].apply(classify)
    states = work["state_raw"].tolist()
    smooth = states[:]
    for i in range(1, len(states) - 1):
        tri = [states[i - 1], states[i], states[i + 1]]
        if tri.count("low") >= 2:
            smooth[i] = "low"
        elif tri.count("high") >= 2:
            smooth[i] = "high"
        else:
            smooth[i] = states[i]
    work["state"] = smooth

    segments = []
    start_idx = 0
    for i in range(1, len(work)):
        if work.loc[i, "state"] != work.loc[start_idx, "state"]:
            segments.append({
                "state": work.loc[start_idx, "state"],
                "start_s": float(work.loc[start_idx, "t_s"]),
                "end_s": float(work.loc[i - 1, "t_s"]),
                "n": i - start_idx,
            })
            start_idx = i
    segments.append({
        "state": work.loc[start_idx, "state"],
        "start_s": float(work.loc[start_idx, "t_s"]),
        "end_s": float(work.loc[len(work) - 1, "t_s"]),
        "n": len(work) - start_idx,
    })

    segments = [s for s in segments if s["state"] in ("low", "high") and s["n"] >= min_stable_samples]

    windows = []
    last_start_s = -1e9
    up_count = 0
    down_count = 0

    for i in range(1, len(segments)):
        prev_seg = segments[i - 1]
        curr_seg = segments[i]
        if prev_seg["state"] == curr_seg["state"]:
            continue
        start_s = curr_seg["start_s"]
        if start_s - last_start_s < min_gap_s:
            continue

        if prev_seg["state"] == "low" and curr_seg["state"] == "high":
            up_count += 1
            windows.append({
                "start_s": start_s,
                "from_m": float(low_target_m),
                "to_m": float(high_target_m),
                "label": f"low_to_high_{up_count}",
            })
            last_start_s = start_s
        elif prev_seg["state"] == "high" and curr_seg["state"] == "low":
            down_count += 1
            windows.append({
                "start_s": start_s,
                "from_m": float(high_target_m),
                "to_m": float(low_target_m),
                "label": f"high_to_low_{down_count}",
            })
            last_start_s = start_s

    return windows

if len(SWITCH_WINDOWS) == 0:
    tmp_windows = auto_detect_switch_windows(
        switch_df,
        LOW_TARGET_M,
        HIGH_TARGET_M,
        signal_col="raw_m",
        min_gap_s=2.0
    )
else:
    tmp_windows = SWITCH_WINDOWS

print(tmp_windows[:10] if len(tmp_windows) else "No windows found")

In [ ]:
# ====== Metric switching ======
def _crossing_time(t, y, threshold, direction="up"):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(t) < 2:
        return np.nan
    for i in range(1, len(y)):
        y0, y1 = y[i - 1], y[i]
        t0, t1 = t[i - 1], t[i]
        if np.isnan(y0) or np.isnan(y1):
            continue
        if direction == "up":
            cond = (y0 < threshold) and (y1 >= threshold)
        else:
            cond = (y0 > threshold) and (y1 <= threshold)
        if cond:
            if y1 == y0:
                return t1
            frac = (threshold - y0) / (y1 - y0)
            return t0 + frac * (t1 - t0)
    return np.nan

def _settling_time(t, y, final_value, tol_m=5.0):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(t) == 0:
        return np.nan
    lower = final_value - tol_m
    upper = final_value + tol_m
    inside = (y >= lower) & (y <= upper)
    for i in range(len(y)):
        if np.all(inside[i:]):
            return float(t[i])
    return np.nan

def analyze_switch_windows(df, windows, signal_col, settling_tol_m=5.0, min_points=3):
    work = df.copy()
    work["t_s"] = pd.to_numeric(work["t_s"], errors="coerce")
    work[signal_col] = pd.to_numeric(work[signal_col], errors="coerce")
    work = work.dropna(subset=["t_s"]).sort_values("t_s").reset_index(drop=True)

    rows = []
    for w in windows:
        start_s = float(w["start_s"])
        from_m = float(w["from_m"])
        to_m = float(w["to_m"])
        label = str(w.get("label", ""))

        seg = work[work["t_s"] >= start_s].copy()
        seg = seg.dropna(subset=[signal_col]).copy()

        if len(seg) < min_points:
            rows.append({
                "label": label,
                "from_m": from_m,
                "to_m": to_m,
                "start_s": start_s,
                "rise_time_10_90_s": np.nan,
                "settling_time_s": np.nan,
                "overshoot_m": np.nan,
                "n_points": len(seg),
            })
            continue

        t = seg["t_s"].to_numpy(dtype=float) - start_s
        y = seg[signal_col].to_numpy(dtype=float)
        delta = to_m - from_m

        if abs(delta) < 1e-9:
            rows.append({
                "label": label,
                "from_m": from_m,
                "to_m": to_m,
                "start_s": start_s,
                "rise_time_10_90_s": 0.0,
                "settling_time_s": 0.0,
                "overshoot_m": 0.0,
                "n_points": len(seg),
            })
            continue

        y10 = from_m + 0.1 * delta
        y90 = from_m + 0.9 * delta

        if delta > 0:
            t10 = _crossing_time(t, y, y10, direction="up")
            t90 = _crossing_time(t, y, y90, direction="up")
            peak = np.nanmax(y)
            overshoot = max(0.0, peak - to_m)
        else:
            t10 = _crossing_time(t, y, y10, direction="down")
            t90 = _crossing_time(t, y, y90, direction="down")
            valley = np.nanmin(y)
            overshoot = max(0.0, to_m - valley)

        rise_time = np.nan if (np.isnan(t10) or np.isnan(t90)) else max(0.0, t90 - t10)
        settle_time = _settling_time(t, y, to_m, tol_m=settling_tol_m)

        rows.append({
            "label": label,
            "from_m": from_m,
            "to_m": to_m,
            "start_s": start_s,
            "rise_time_10_90_s": rise_time,
            "settling_time_s": settle_time,
            "overshoot_m": overshoot,
            "n_points": len(seg),
        })

    return pd.DataFrame(rows)

def score_switch(baseline_metrics_df, ab_metrics_df, tracker_gate_reject_ratio):
    a_rise = safe_mean(ab_metrics_df["rise_time_10_90_s"]) if len(ab_metrics_df) else np.nan
    a_settle = safe_mean(ab_metrics_df["settling_time_s"]) if len(ab_metrics_df) else np.nan
    a_overshoot = safe_mean(ab_metrics_df["overshoot_m"]) if len(ab_metrics_df) else np.nan

    if np.isnan(a_rise):
        a_rise = 1e6
    if np.isnan(a_settle):
        a_settle = 1e6
    if np.isnan(a_overshoot):
        a_overshoot = 1e6
    if np.isnan(tracker_gate_reject_ratio):
        tracker_gate_reject_ratio = 1.0

    return float(
        1.0 * a_rise +
        0.8 * a_settle +
        2.0 * tracker_gate_reject_ratio +
        0.2 * a_overshoot
    )

# test nhanh
tmp_test = analyze_switch_windows(
    switch_df,
    tmp_windows,
    signal_col="baseline_m",
    settling_tol_m=SWITCH_SETTLING_TOL_M
)
display(tmp_test.head())

In [ ]:
# ====== Sweep switching để tạo switch_tuning_df ======
topk_static = (
    res_df.sort_values(
        ["score_static", "ab_sigma", "ab_rmse", "ab_rate_std", "ab_tracker_gate_reject_ratio"]
    )
    .head(TOP_K_STATIC)
    .reset_index(drop=True)
)

print("Top-K static configs:")
display(topk_static)

switch_rows = []

for _, row in topk_static.iterrows():
    a = float(row["alpha"])
    b = float(row["beta"])

    for g in SWITCH_GATE_LIST:
        for mr in SWITCH_MAX_REJECT_LIST:
            cfg = ABConfig(
                alpha=a,
                beta=b,
                gate_threshold_m=float(g),
                max_reject=int(mr),
            )

            sw = run_alpha_beta(switch_df, cfg)

            baseline_metrics_tmp = analyze_switch_windows(
                sw,
                tmp_windows,
                signal_col="baseline_m",
                settling_tol_m=SWITCH_SETTLING_TOL_M,
            )

            ab_metrics_tmp = analyze_switch_windows(
                sw,
                tmp_windows,
                signal_col="ab_est_m",
                settling_tol_m=SWITCH_SETTLING_TOL_M,
            )

            tracker_gate_reject_ratio = float(
                np.nanmean(pd.to_numeric(sw["ab_rejected_by_gate"], errors="coerce").fillna(0))
            )
            sensor_reject_ratio = float((pd.to_numeric(sw["reject_reason"], errors="coerce").fillna(0) != 0).mean())

            switch_rows.append({
                "alpha": a,
                "beta": b,
                "gate_m": float(g),
                "max_reject": int(mr),
                "baseline_mean_rise_s": safe_mean(baseline_metrics_tmp["rise_time_10_90_s"]) if len(baseline_metrics_tmp) else np.nan,
                "baseline_mean_settle_s": safe_mean(baseline_metrics_tmp["settling_time_s"]) if len(baseline_metrics_tmp) else np.nan,
                "ab_mean_rise_s": safe_mean(ab_metrics_tmp["rise_time_10_90_s"]) if len(ab_metrics_tmp) else np.nan,
                "ab_mean_settle_s": safe_mean(ab_metrics_tmp["settling_time_s"]) if len(ab_metrics_tmp) else np.nan,
                "ab_mean_overshoot_m": safe_mean(ab_metrics_tmp["overshoot_m"]) if len(ab_metrics_tmp) else np.nan,
                "sensor_reject_ratio": sensor_reject_ratio,
                "ab_tracker_gate_reject_ratio": tracker_gate_reject_ratio,
                "score_switch": score_switch(
                    baseline_metrics_tmp,
                    ab_metrics_tmp,
                    tracker_gate_reject_ratio
                ),
            })

switch_tuning_df = pd.DataFrame(switch_rows)

if len(switch_tuning_df) == 0:
    raise RuntimeError("switch_tuning_df rỗng. Hãy kiểm tra topk_static, tmp_windows hoặc analyze_switch_windows().")

switch_tuning_df = switch_tuning_df.sort_values(
    ["score_switch", "ab_tracker_gate_reject_ratio", "ab_mean_rise_s", "ab_mean_settle_s"]
).reset_index(drop=True)

display(switch_tuning_df.head(20))

In [ ]:
# ====== Chọn cấu hình cuối từ switching ======
if "switch_tuning_df" not in globals():
    raise RuntimeError("switch_tuning_df chưa được tạo. Hãy chạy cell sweep switching phía trên trước.")

if switch_tuning_df is None or len(switch_tuning_df) == 0:
    raise RuntimeError("switch_tuning_df đang rỗng. Hãy kiểm tra lại switch_rows / tmp_windows / cell sweep switching.")

best_switch = switch_tuning_df.iloc[0]

FINAL_ALPHA = float(best_switch["alpha"])
FINAL_BETA = float(best_switch["beta"])
FINAL_GATE = float(best_switch["gate_m"])
FINAL_MAX_REJECT = int(best_switch["max_reject"])

print("Best after switching sweep:")
print(f"alpha = {FINAL_ALPHA:.3f}, beta = {FINAL_BETA:.3f}, gate = {FINAL_GATE:.1f}, max_reject = {FINAL_MAX_REJECT}")

final_cfg = ABConfig(
    alpha=FINAL_ALPHA,
    beta=FINAL_BETA,
    gate_threshold_m=FINAL_GATE,
    max_reject=FINAL_MAX_REJECT,
)

switch_ab = run_alpha_beta(switch_df, final_cfg)

if len(SWITCH_WINDOWS) == 0:
    SWITCH_WINDOWS = auto_detect_switch_windows(
        switch_ab,
        LOW_TARGET_M,
        HIGH_TARGET_M,
        signal_col="raw_m",
        min_gap_s=2.0
    )

pd.DataFrame(SWITCH_WINDOWS)

In [ ]:
# ====== Summary switching + plots ======
baseline_switch_metrics = analyze_switch_windows(
    switch_df,
    SWITCH_WINDOWS,
    signal_col="baseline_m",
    settling_tol_m=SWITCH_SETTLING_TOL_M,
)

ab_switch_metrics = analyze_switch_windows(
    switch_ab,
    SWITCH_WINDOWS,
    signal_col="ab_est_m",
    settling_tol_m=SWITCH_SETTLING_TOL_M,
)

switch_summary = pd.DataFrame([{
    "n_samples": len(switch_df),
    "mean_fps": safe_mean(switch_df["fps"]) if "fps" in switch_df.columns else np.nan,
    "baseline_mean_rise_s": safe_mean(baseline_switch_metrics["rise_time_10_90_s"]),
    "baseline_mean_settle_s": safe_mean(baseline_switch_metrics["settling_time_s"]),
    "ab_mean_rise_s": safe_mean(ab_switch_metrics["rise_time_10_90_s"]),
    "ab_mean_settle_s": safe_mean(ab_switch_metrics["settling_time_s"]),
    "sensor_reject_ratio": float((switch_df["reject_reason"] != 0).mean()),
    "ab_tracker_gate_reject_ratio": float((switch_ab["ab_rejected_by_gate"] != 0).mean()),
}])

display(baseline_switch_metrics)
display(ab_switch_metrics)
display(switch_summary)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(switch_ab["t_s"], switch_ab["raw_m"], label="raw")
axes[0].plot(switch_ab["t_s"], switch_ab["baseline_m"], label="baseline")
axes[0].plot(switch_ab["t_s"], switch_ab["ab_est_m"], label="alpha-beta")
axes[0].axhline(LOW_TARGET_M, linestyle="--", label="low target")
axes[0].axhline(HIGH_TARGET_M, linestyle="--", label="high target")
axes[0].set_title("Switching: raw vs baseline vs alpha-beta")
axes[0].set_ylabel("Distance (m)")
axes[0].legend()

axes[1].step(switch_ab["t_s"], switch_ab["reject_reason"], where="post")
axes[1].set_title("Switching: sensor-side reject reason")
axes[1].set_ylabel("reject_reason")

axes[2].step(switch_ab["t_s"], switch_ab["ab_rejected_by_gate"], where="post")
axes[2].set_title("Switching: tracker-side gate reject")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("ab_rejected_by_gate")

plt.tight_layout()
plt.show()

In [ ]:
# ====== Reject reason tables ======
reject_reason_map = {
    0: "NONE",
    1: "BAD_FRAME",
    2: "BAD_CRC",
    3: "DATA_INVALID",
    4: "BLIND_AREA",
    5: "OUT_OF_RANGE",
}

def reject_reason_table(df, tag):
    tmp = (
        pd.to_numeric(df["reject_reason"], errors="coerce")
        .fillna(0)
        .astype(int)
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("reject_reason")
        .reset_index(name="count")
    )
    tmp["label"] = tmp["reject_reason"].map(reject_reason_map).fillna("UNKNOWN")
    tmp["tag"] = tag
    tmp["ratio"] = tmp["count"] / len(df)
    return tmp

display(reject_reason_table(tripod_df_raw, "tripod"))
display(reject_reason_table(switch_df_raw, "switching"))

In [ ]:
# ====== Export kết quả ======
output_dir = Path("analysis_outputs")
output_dir.mkdir(exist_ok=True)

baseline_summary_tripod.to_csv(output_dir / "baseline_summary_tripod.csv", index=False)
tripod_ab_compare.to_csv(output_dir / "tripod_ab_compare.csv", index=False)
res_df.to_csv(output_dir / "alphabeta_grid_results.csv", index=False)
baseline_switch_metrics.to_csv(output_dir / "switch_metrics_baseline.csv", index=False)
ab_switch_metrics.to_csv(output_dir / "switch_metrics_alphabeta.csv", index=False)
switch_summary.to_csv(output_dir / "switch_summary.csv", index=False)
reject_reason_table(tripod_df_raw, "tripod").to_csv(output_dir / "tripod_reject_reason_counts.csv", index=False)
reject_reason_table(switch_df_raw, "switching").to_csv(output_dir / "switch_reject_reason_counts.csv", index=False)
switch_tuning_df.to_csv(output_dir / "switch_tuning_results.csv", index=False)

print("Saved to:", output_dir.resolve())
print(sorted(os.listdir(output_dir)))